In [91]:
from src.Features import selecting_data
from sklearn.model_selection import train_test_split,GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from src.Data_utils import Calculate_metrics
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor


data=selecting_data()

print(data)

(       MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0      8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1      8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2      7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3      5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4      3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   
...       ...       ...       ...        ...         ...       ...       ...   
20635  1.5603      25.0  5.045455   1.133333       845.0  2.560606     39.48   
20636  2.5568      18.0  6.114035   1.315789       356.0  3.122807     39.49   
20637  1.7000      17.0  5.205543   1.120092      1007.0  2.325635     39.43   
20638  1.8672      18.0  5.329513   1.171920       741.0  2.123209     39.43   
20639  2.3886      16.0  5.254717   1.162264      1387.0  2.616981     39.37   

       Longitude  
0        -122.23  


In [92]:
##Splitting into training and testing sets
X_train,X_test,Y_train,Y_test=train_test_split(data[0],data[1], test_size=0.2, random_state=42)

##Selecting features from feature selection step

X_train=X_train[['MedInc', 'Latitude', 'AveOccup', 'Longitude']]
X_test=X_test[['MedInc', 'Latitude', 'AveOccup', 'Longitude']]


In [93]:
#Linear model is just used as benchmark

linear_model=LinearRegression()
linear_model.fit(X_train,Y_train)
Y_pred=linear_model.predict(X_test)

print(Calculate_metrics(Y_test,Y_pred))

{'RMSE': 0.131805143041405, 'MAE': 0.27505126456318224, 'MAPE': 141.44802214999726}


In [ ]:
##Support vector regression

##First scaling and centering of features is an assumption for the model.
##It is necessary to scale and center after the split to not contaminate the testing set.

scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

param_grid = {
    'kernel': ['linear',"rbf"], ##Controls the shape of the decision boundary rbf and poly can fit more complex decision boundaries
    'C': [0.1, 1,10], ##Regularization parameter on the loss function
    'epsilon': [0.01, 0.1] ,## epsilon insensitive tube defines the range where errors are ignored
    'gamma': ['scale']
}

##Compute grid search over all combinations of hyperparameters

grid_search = GridSearchCV(
    SVR(),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_scaled,Y_train)
print(grid_search.best_params_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [87]:
Model=SVR(kernel='rbf',C=10,epsilon=0.1,gamma="scale")
Model.fit(X_train_scaled,Y_train)
SVR_pred=Model.predict(X_test_scaled)
Calculate_metrics(Y_test,SVR_pred)

{'RMSE': 0.07883480204862274,
 'MAE': 0.20046796843988554,
 'MAPE': 102.717346067208}

In [89]:
##Ensemble methods

##XGBOOST

parameter_grid_XGBOOST={
    'n_estimators': [100, 200, 500],         # Number of boosting rounds / trees
    'max_depth': [3, 5, 7, 9],               # Maximum depth of each tree
    'learning_rate': [0.01, 0.05, 0.1, 0.2],# Step size shrinkage
    'subsample': [0.6, 0.8, 1.0],            # Fraction of samples per tree
    'reg_alpha': [0, 0.01, 0.1],             # L1 regularization
    'reg_lambda': [1, 1.5, 2]                # L2 regularization
}



random_search_XGBOOST = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=parameter_grid_XGBOOST,
    n_iter=200,                     # Try 200 random combinations
    cv=5,
    scoring='neg_root_mean_squared_error', ##Minimizing root mean square error
    n_jobs=-1,
    verbose=1
)

random_search_XGBOOST.fit(X_train, Y_train)
print(random_search_XGBOOST.best_params_)


NameError: name 'XGBRegressor' is not defined